# 17 · Gray-Box AIA72 Cross-Cycle Reliability Audit

**Purpose:** quantify how the already-frozen 72-hour SHARP, AIA, equal-weight fusion, conformal uncertainty, and fallback-first v3 trust signals change from earlier Solar Cycle 24 development support to Solar Cycle 25 and supplementary 2026.

This notebook performs **no model fitting, no threshold tuning, no policy selection, and no policy redesign**. It is a post-hoc reliability/domain-shift audit using frozen predictions and frozen thresholds.

### Important evidence boundary
- The canonical multimodal master contains only **non-training** predictions. Therefore 2010–2013 training cases are intentionally not scored here.
- Earlier Cycle-24 roles include data already used for model validation, calibration, conformal calibration, or policy validation; their results are **development/descriptive**, not independent test evidence.
- Cycle 25 was subsequently used as development-diagnostic evidence before v3 was frozen.
- Supplementary 2026 was the one-shot clean future evaluation for v3 and is now already spent; this notebook does not redesign anything from it.
- Existing 48-hour AIA labels/results remain separate and are not used as 72-hour truth.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.linear_model import LogisticRegression
except Exception:
    LogisticRegression = None

# ============================================================
# Frozen inputs
# ============================================================
ROOT = Path('/home/abmoses2000')
MASTER = ROOT/'graybox_aia72_master_predictions_v2_20261005'/'graybox_aia72_master_predictions.csv.gz'
MASTER_SHA = '17300093b88b7c61ecadf5eb28acba0d6d023b340d7740807852dd4352b22b46'
SHARP = ROOT/'sharp72_predictions_frozen_20261002.csv.gz'
SHARP_SHA = '7368c002a978321cdc67221564f3b7901687badc7aba28ab9df615179d1e017b'
MOND_THRESHOLDS = ROOT/'graybox_aia72_conformal_mondrian_v2_20261005'/'mondrian_thresholds.csv'
CUTS = ROOT/'graybox_aia72_policy_freeze_v1_20261005'/'candidate_cutoffs.csv'
ALARM_THRESHOLDS = ROOT/'graybox_aia72_alarm_and_primary_policy_freeze_v1_20261005'/'alarm_thresholds.csv'
V3_PROTOCOL = ROOT/'graybox_aia72_policy_v3_fallback_first_freeze_20261005'/'protocol.json'

OUT = ROOT/'graybox_aia72_cross_cycle_reliability_audit_20261006'
if OUT.exists():
    raise SystemExit(f'Refusing to overwrite existing output: {OUT}')
(OUT/'figures').mkdir(parents=True)

SEEDS = [17,29,43]
PRIMARY_ALPHA = 0.10
PRIMARY_PCT = 0.90

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

def require(cond,msg):
    if not cond: raise ValueError(msg)

for p in [MASTER,SHARP,MOND_THRESHOLDS,CUTS,ALARM_THRESHOLDS,V3_PROTOCOL]:
    require(p.is_file(), f'Missing frozen input: {p}')
require(sha256(MASTER)==MASTER_SHA,'Master SHA changed')
require(sha256(SHARP)==SHARP_SHA,'SHARP SHA changed')

v3 = json.loads(V3_PROTOCOL.read_text())
require(v3['primary_policy_id']=='fallback_v3_q90','v3 primary policy changed')

# ============================================================
# Helpers
# ============================================================
def entropy_binary(p):
    q=np.clip(np.asarray(p,float),1e-12,1-1e-12)
    return -(q*np.log(q)+(1-q)*np.log(1-q))

def binary_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    tp=int(((y==1)&(pred==1)).sum()); fp=int(((y==0)&(pred==1)).sum())
    fn=int(((y==1)&(pred==0)).sum()); tn=int(((y==0)&(pred==0)).sum())
    recall=tp/(tp+fn) if tp+fn else np.nan
    spec=tn/(tn+fp) if tn+fp else np.nan
    prec=tp/(tp+fp) if tp+fp else np.nan
    f1=2*prec*recall/(prec+recall) if np.isfinite(prec) and np.isfinite(recall) and prec+recall else np.nan
    tss=recall+spec-1 if np.isfinite(recall) and np.isfinite(spec) else np.nan
    denom=((tp+fn)*(fn+tn)+(tp+fp)*(fp+tn))
    hss=2*(tp*tn-fn*fp)/denom if denom else np.nan
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,recall=recall,specificity=spec,precision=prec,f1=f1,tss=tss,hss=hss)

def brier(y,p):
    y=np.asarray(y,int); p=np.asarray(p,float); m=np.isfinite(p)
    return float(np.mean((p[m]-y[m])**2)) if m.any() else np.nan

def logloss(y,p):
    y=np.asarray(y,int); p=np.asarray(p,float); m=np.isfinite(p)
    if not m.any(): return np.nan
    q=np.clip(p[m],1e-12,1-1e-12); yy=y[m]
    return float(-np.mean(yy*np.log(q)+(1-yy)*np.log(1-q)))

def ece_equal_width(y,p,bins=10):
    y=np.asarray(y,int); p=np.asarray(p,float); m=np.isfinite(p)
    y=y[m]; p=p[m]
    edges=np.linspace(0,1,bins+1)
    ids=np.minimum(np.digitize(p,edges[1:-1],right=False),bins-1)
    ece=0.0; rows=[]
    for b in range(bins):
        z=ids==b
        if not z.any():
            rows.append((b,edges[b],edges[b+1],0,np.nan,np.nan))
            continue
        mp=float(p[z].mean()); oy=float(y[z].mean()); n=int(z.sum())
        ece += (n/len(y))*abs(mp-oy)
        rows.append((b,edges[b],edges[b+1],n,mp,oy))
    return float(ece), rows

def calibration_slope_intercept(y,p):
    if LogisticRegression is None: return np.nan,np.nan
    y=np.asarray(y,int); p=np.asarray(p,float); m=np.isfinite(p)
    if m.sum()<10 or len(np.unique(y[m]))<2: return np.nan,np.nan
    q=np.clip(p[m],1e-6,1-1e-6)
    x=np.log(q/(1-q)).reshape(-1,1)
    lr=LogisticRegression(C=1e6,solver='lbfgs',max_iter=5000)
    lr.fit(x,y[m])
    return float(lr.coef_[0,0]), float(lr.intercept_[0])

def mondrian_membership(p,q0,q1):
    p=np.asarray(p,float); finite=np.isfinite(p)
    inc0=finite&(p<=q0); inc1=finite&((1-p)<=q1)
    return inc0,inc1

# ============================================================
# Load and align frozen predictions
# ============================================================
master=pd.read_csv(MASTER,low_memory=False)
sharp=pd.read_csv(SHARP,low_memory=False)
master['forecast_case_id']=master['forecast_case_id'].astype(str)
sharp['forecast_case_id']=sharp['forecast_case_id'].astype(str)
require(master['forecast_case_id'].is_unique,'Duplicate master IDs')
require(sharp['forecast_case_id'].is_unique,'Duplicate SHARP IDs')

m=master.merge(sharp,on='forecast_case_id',how='inner',suffixes=('_aia','_sharp'),validate='one_to_one')
require(len(m)==len(master),f'Alignment lost rows: master={len(master)} merged={len(m)}')

label_a='label_aia' if 'label_aia' in m.columns else 'label'
label_s='label_sharp' if 'label_sharp' in m.columns else label_a
y_a=pd.to_numeric(m[label_a],errors='raise').astype(int).to_numpy()
y_s=pd.to_numeric(m[label_s],errors='raise').astype(int).to_numpy()
require(np.array_equal(y_a,y_s),'AIA/SHARP labels disagree')
m['label_final']=y_a

issue_col='issue_utc_aia' if 'issue_utc_aia' in m.columns else ('issue_utc' if 'issue_utc' in m.columns else None)
require(issue_col is not None,'Missing issue time')
m['issue_dt']=pd.to_datetime(m[issue_col],utc=True,errors='raise')
m['year']=m['issue_dt'].dt.year.astype(int)
role_col='role_aia' if 'role_aia' in m.columns else ('role' if 'role' in m.columns else None)
require(role_col is not None,'Missing role')
m['role_final']=m[role_col].astype(str)

m['aia_probability']=pd.to_numeric(m['probability'],errors='coerce')
m['sharp_probability']=pd.to_numeric(m['probability_gru_mean'],errors='coerce')
m['fusion_probability']=0.5*m['aia_probability']+0.5*m['sharp_probability']

aia_seed=m[[f'probability_seed_{z}_aia' if f'probability_seed_{z}_aia' in m.columns else f'probability_seed_{z}' for z in SEEDS]].astype(float).to_numpy()
sharp_seed=m[[f'probability_seed_{z}_sharp' if f'probability_seed_{z}_sharp' in m.columns else f'probability_seed_{z}' for z in SEEDS]].astype(float).to_numpy()
fusion_seed=0.5*aia_seed+0.5*sharp_seed
m['aia_seed_std']=np.nanstd(aia_seed,axis=1,ddof=1)
m['sharp_seed_std']=np.nanstd(sharp_seed,axis=1,ddof=1)
m['fusion_seed_std']=np.nanstd(fusion_seed,axis=1,ddof=1)
m['sharp_aia_probability_gap']=np.abs(m['sharp_probability']-m['aia_probability'])
m['fusion_entropy']=entropy_binary(m['fusion_probability'])

# Historical evidence strata. These are provenance/evidence labels, not newly claimed independent splits.
def regime(row):
    r=row['role_final']; y=row['year']
    if r=='retrospective_cycle25': return 'cycle25_2021_2025_diagnostic'
    if r=='supplementary_2026': return 'supplementary_2026_locked'
    if y<=2019: return 'cycle24_earlier_development'
    return 'other_or_gap'
m['reliability_regime']=m.apply(regime,axis=1)

# ============================================================
# Frozen alarm / conformal / v3 definitions
# ============================================================
alarm=pd.read_csv(ALARM_THRESHOLDS)
alarm_map={r.branch:float(r.threshold) for r in alarm.itertuples()}
require(set(alarm_map)=={'sharp','aia','sharp_aia'},'Alarm branches changed')

mond=pd.read_csv(MOND_THRESHOLDS)
q={}
for b in ['sharp','aia','sharp_aia']:
    z=mond[(mond['branch']==b)&np.isclose(mond['alpha'].astype(float),PRIMARY_ALPHA)]
    require(len(z)==1,f'Missing Mondrian threshold for {b}')
    q[b]=(float(z.iloc[0]['qhat_class0']),float(z.iloc[0]['qhat_class1']))

cuts=pd.read_csv(CUTS)
def cutoff(metric,pct=PRIMARY_PCT):
    z=cuts[(cuts['metric']==metric)&np.isclose(cuts['percentile'].astype(float),pct)]
    require(len(z)==1,f'Missing cutoff {metric} q{pct}')
    return float(z.iloc[0]['cutoff'])

fusion_seed_cut=cutoff('fusion_seed_std')
gap_cut=cutoff('sharp_aia_probability_gap')
entropy_cut=cutoff('fusion_entropy')

for branch,pcol in [('sharp','sharp_probability'),('aia','aia_probability'),('sharp_aia','fusion_probability')]:
    inc0,inc1=mondrian_membership(m[pcol].to_numpy(float),*q[branch])
    m[f'{branch}_inc0']=inc0; m[f'{branch}_inc1']=inc1
    yy=m['label_final'].to_numpy(int)
    m[f'{branch}_covered']=np.where(yy==1,inc1,inc0)

aia_status_col='input_status' if 'input_status' in m.columns else ('input_status_aia' if 'input_status_aia' in m.columns else None)
aia_ok=np.isfinite(m['aia_probability'].to_numpy(float))
if aia_status_col is not None:
    aia_ok &= (m[aia_status_col].astype(str).to_numpy()=='ok')
sharp_ok=np.isfinite(m['sharp_probability'].to_numpy(float))
fusion_ok=aia_ok&sharp_ok&np.isfinite(m['fusion_probability'].to_numpy(float))
fusion_single=((m['sharp_aia_inc0']&~m['sharp_aia_inc1'])|(~m['sharp_aia_inc0']&m['sharp_aia_inc1'])).to_numpy(bool)
normal=(fusion_ok & fusion_single &
        (m['fusion_seed_std'].to_numpy(float)<=fusion_seed_cut) &
        (m['sharp_aia_probability_gap'].to_numpy(float)<=gap_cut) &
        (m['fusion_entropy'].to_numpy(float)<=entropy_cut))
degraded=(~normal)&sharp_ok
abstain=~(normal|degraded)
m['v3_q90_state']=np.where(normal,'NORMAL',np.where(degraded,'DEGRADED','ABSTAIN'))

# ============================================================
# Reliability summaries by regime, role and year
# ============================================================
def summarize_group(g,group_name,group_value):
    y=g['label_final'].to_numpy(int)
    out=[]
    prevalence=float(y.mean()) if len(y) else np.nan
    for branch,pcol in [('sharp','sharp_probability'),('aia','aia_probability'),('sharp_aia','fusion_probability')]:
        p=g[pcol].to_numpy(float); avail=np.isfinite(p)
        yy=y[avail]; pp=p[avail]
        pred=(pp>=alarm_map[branch]).astype(int)
        bm=binary_metrics(yy,pred) if len(yy) else {k:np.nan for k in ['tp','fp','fn','tn','recall','specificity','precision','f1','tss','hss']}
        ece,bins=ece_equal_width(yy,pp,10) if len(yy) else (np.nan,[])
        slope,intercept=calibration_slope_intercept(yy,pp)
        cov=g.loc[avail,f'{branch}_covered'].astype(bool).to_numpy()
        pos=(yy==1); neg=(yy==0)
        out.append({
            'grouping':group_name,'group':str(group_value),'branch':branch,
            'cases':int(len(g)),'available_cases':int(avail.sum()),'positives':int(y.sum()),'prevalence':prevalence,
            'brier':brier(yy,pp),'log_loss':logloss(yy,pp),'ece_10bin':ece,
            'calibration_slope':slope,'calibration_intercept':intercept,
            'overall_conformal_coverage':float(cov.mean()) if len(cov) else np.nan,
            'flare_conformal_coverage':float(cov[pos].mean()) if pos.any() else np.nan,
            'no_flare_conformal_coverage':float(cov[neg].mean()) if neg.any() else np.nan,
            **bm
        })
    return out

rows=[]
for value,g in m.groupby('reliability_regime',sort=False): rows += summarize_group(g,'regime',value)
for value,g in m.groupby('role_final',sort=False): rows += summarize_group(g,'role',value)
for value,g in m.groupby('year',sort=True): rows += summarize_group(g,'year',value)
metrics=pd.DataFrame(rows)
metrics.to_csv(OUT/'cross_cycle_branch_metrics.csv',index=False)

# Trust-signal / state summary
state_rows=[]
for value,g in m.groupby('reliability_regime',sort=False):
    n=len(g); positives=int(g['label_final'].sum())
    counts=g['v3_q90_state'].value_counts()
    state_rows.append({
        'regime':value,'cases':n,'positives':positives,'prevalence':positives/n if n else np.nan,
        'mean_sharp_aia_gap':float(g['sharp_aia_probability_gap'].mean()),
        'median_sharp_aia_gap':float(g['sharp_aia_probability_gap'].median()),
        'mean_fusion_seed_std':float(g['fusion_seed_std'].mean()),
        'mean_fusion_entropy':float(g['fusion_entropy'].mean()),
        'normal_cases':int(counts.get('NORMAL',0)),
        'degraded_cases':int(counts.get('DEGRADED',0)),
        'abstain_cases':int(counts.get('ABSTAIN',0)),
        'normal_rate':float((g['v3_q90_state']=='NORMAL').mean()),
        'degraded_rate':float((g['v3_q90_state']=='DEGRADED').mean()),
        'abstain_rate':float((g['v3_q90_state']=='ABSTAIN').mean()),
    })
states=pd.DataFrame(state_rows)
states.to_csv(OUT/'cross_cycle_trust_state_summary.csv',index=False)

# Explicit support/gap table so the paper does not imply predictions where none exist.
support=pd.DataFrame([
    {'period':'2010-2013','status':'training period absent from canonical non-training master','use':'do not score cross-cycle reliability from in-sample predictions'},
    {'period':'2014-2019','status':'available across earlier non-training development/calibration/policy roles','use':'development/descriptive Cycle-24 reliability only'},
    {'period':'2020','status':'not represented in accepted multimodal prediction package','use':'report as support gap; do not interpolate'},
    {'period':'2021-2025','status':'retrospective_cycle25 predictions available','use':'diagnostic/post-hoc for v3 because Cycle25 informed redesign'},
    {'period':'2026','status':'supplementary_2026 predictions available','use':'reproduce frozen clean-v3 result; no redesign'},
])
support.to_csv(OUT/'cross_cycle_support_boundaries.csv',index=False)

# ============================================================
# Figures
# ============================================================
order=['cycle24_earlier_development','cycle25_2021_2025_diagnostic','supplementary_2026_locked']
plot=metrics[(metrics['grouping']=='regime') & metrics['group'].isin(order)].copy()
plot['group']=pd.Categorical(plot['group'],categories=order,ordered=True)
plot=plot.sort_values(['group','branch'])

for metric,title,filename,ylim in [
    ('flare_conformal_coverage','Flare-class conformal coverage across temporal regimes','fig1_flare_coverage_by_regime.png',(0,1)),
    ('brier','Brier score across temporal regimes','fig2_brier_by_regime.png',None),
    ('tss','TSS across temporal regimes','fig3_tss_by_regime.png',None),
    ('ece_10bin','Expected calibration error across temporal regimes','fig4_ece_by_regime.png',None),
]:
    piv=plot.pivot(index='group',columns='branch',values=metric)
    ax=piv.plot(kind='bar',figsize=(10,5))
    if metric=='flare_conformal_coverage': ax.axhline(0.90,linestyle='--',linewidth=1)
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(metric)
    if ylim: ax.set_ylim(*ylim)
    plt.xticks(rotation=15,ha='right'); plt.tight_layout(); plt.savefig(OUT/'figures'/filename,dpi=220); plt.close()

ax=states.set_index('regime')[['normal_rate','degraded_rate','abstain_rate']].loc[[x for x in order if x in states['regime'].values]].plot(kind='bar',stacked=True,figsize=(10,5))
ax.set_title('Post-hoc application of frozen v3 q90 state logic across regimes')
ax.set_ylabel('Fraction of cases'); ax.set_xlabel(''); ax.set_ylim(0,1)
plt.xticks(rotation=15,ha='right'); plt.tight_layout(); plt.savefig(OUT/'figures'/'fig5_v3_state_mix_by_regime.png',dpi=220); plt.close()

# ============================================================
# Protocol / manuscript-safe summary
# ============================================================
protocol={
    'status':'POSTHOC_CROSS_CYCLE_RELIABILITY_AUDIT_COMPLETE',
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'horizon_hours':72,
    'new_model_fitting':False,
    'new_threshold_tuning':False,
    'new_policy_selection':False,
    'new_policy_redesign':False,
    'primary_policy':'fallback_v3_q90',
    'evidence_boundary':{
        'cycle24':'development/descriptive only; contains prior model/calibration/policy development roles',
        'cycle25':'diagnostic/post-hoc for v3 because Cycle25 informed v3 redesign',
        'supplementary_2026':'reproduction/stratification of already-spent clean future evaluation; no redesign allowed',
    },
    'source_sha256':{'master_predictions':MASTER_SHA,'sharp_predictions':SHARP_SHA},
    'outputs':['cross_cycle_branch_metrics.csv','cross_cycle_trust_state_summary.csv','cross_cycle_support_boundaries.csv','figures/'],
}
(OUT/'protocol.json').write_text(json.dumps(protocol,indent=2)+'\n')

print('===== CROSS-CYCLE RELIABILITY AUDIT COMPLETE =====')
print('rows aligned:',len(m))
print('\nSupport by regime:')
print(m.groupby('reliability_regime').agg(cases=('forecast_case_id','size'),positives=('label_final','sum'),first=('issue_dt','min'),last=('issue_dt','max')).to_string())
print('\nBranch metrics by regime:')
print(plot[['group','branch','cases','positives','brier','ece_10bin','calibration_slope','calibration_intercept','flare_conformal_coverage','tss','recall','specificity']].round(6).to_string(index=False))
print('\nFrozen-v3 post-hoc state mix:')
print(states.round(6).to_string(index=False))
print('\nOutputs:',OUT)


## Interpretation rules after execution

1. **Do not call Cycle-24 metrics an independent test.** They summarize earlier development/calibration support.
2. **Do not call Cycle-25 v3 performance independent.** Cycle 25 informed the v3 redesign.
3. **Do not retune from 2026.** The supplementary-2026 test is already spent.
4. Use this notebook to quantify temporal reliability shift and motivate the next phases: complete calibration diagnostics, multimodal OOD/applicability, AR/event-disjoint analysis, historical outage replay, fallback completion, and rolling operational replay.
5. The absence of 2010–2013 from this score table is intentional: those rows belong to training support and the canonical multimodal master contains non-training predictions only.
